In [1]:
# Test data repair scripts for NY

import sys
import os
from dotenv import load_dotenv, find_dotenv
import matplotlib.pyplot as plt
import time
import pandas as pd
import numpy as np
import json

load_dotenv(find_dotenv())

ROOT_PATH = os.getenv("ROOT_PATH")
MY_DATA_PATH = os.getenv("MY_DATA_PATH")
RAW_DATA_PATH = os.getenv("RAW_DATA_PATH")
DEWEY_PATH = os.path.join(RAW_DATA_PATH, "dewey-downloads", "building-permits-united-states")

sys.path.append(os.path.join(ROOT_PATH, "scripts"))
import data_utils as du

sys.path.append(os.path.join(ROOT_PATH, "agent/scripts"))

from phi_data_repair import data_repair
MY_JURISDICTION = "Philadelphia"

INPUT_FILEPATH = os.path.join(MY_DATA_PATH, "processed_data", "permits_top50_sample.parquet")


In [2]:
df = pd.read_parquet(INPUT_FILEPATH)
sub_df = df[df["JURISDICTION"] == MY_JURISDICTION]

for col in ['FILE_DATE', 'PERMIT_DATE', 'FINAL_DATE']:
    sub_df[f'{col}_FLAG'] = ""

#sub_df_filled = sub_df.copy()
sub_df_filled = data_repair(sub_df)

assert(len(sub_df) == len(sub_df_filled))


In [3]:
print(f"FILE_DATE available (all): {sub_df['FILE_DATE'].notna().mean():.1%} -> {sub_df_filled['FILE_DATE'].notna().mean():.1%}")

print(f"PERMIT_DATE available (all): {sub_df['PERMIT_DATE'].notna().mean():.1%} -> {sub_df_filled['PERMIT_DATE'].notna().mean():.1%}")

print(f"FINAL_DATE available (all): {sub_df['FINAL_DATE'].notna().mean():.1%} -> {sub_df_filled['FINAL_DATE'].notna().mean():.1%}")

mask1 = sub_df['STATUS_NORMALIZED'].isin(['Active', 'Final'])
mask2 = sub_df_filled['STATUS_NORMALIZED'].isin(['Active', 'Final'])
print(f"PERMIT_DATE available (active/final): {sub_df.loc[mask1]['PERMIT_DATE'].notna().mean():.1%} -> {sub_df_filled.loc[mask2]['PERMIT_DATE'].notna().mean():.1%}")

mask1 = sub_df['STATUS_NORMALIZED'].isin(['Final'])
mask2 = sub_df_filled['STATUS_NORMALIZED'].isin(['Final'])
print(f"FINAL_DATE available (final): {sub_df.loc[mask1]['FINAL_DATE'].notna().mean():.1%} -> {sub_df_filled.loc[mask2]['FINAL_DATE'].notna().mean():.1%}")


FILE_DATE available (all): 17.7% -> 17.7%
PERMIT_DATE available (all): 96.1% -> 99.2%
FINAL_DATE available (all): 81.9% -> 82.2%
PERMIT_DATE available (active/final): 99.9% -> 99.9%
FINAL_DATE available (final): 97.5% -> 97.6%


In [4]:
print(sub_df['STATUS_NORMALIZED'].value_counts())
print(sub_df_filled['STATUS_NORMALIZED'].value_counts())

STATUS_NORMALIZED
Final        1485
Active        244
Inactive      191
In Review       7
Name: count, dtype: int64
STATUS_NORMALIZED
Final        1523
Active        265
Inactive      200
In Review      10
Name: count, dtype: int64


In [18]:
#mask = sub_df_filled["FILE_DATE"].isna() 
mask = sub_df_filled["JURISDICTION"].notna()
sample = sub_df_filled.loc[mask].sample(1).iloc[0]
DATA = sample["DATA"]
DATES_DATA = du.extract_date_fields(DATA) 

print(f"STATUS_NORMALIZED: {sample['STATUS_NORMALIZED']}    *Filled: {sample['STATUS_NORMALIZED_FLAG']}*")
print(f"RECORD_TYPE_ORIGINAL: {sample['RECORD_TYPE_ORIGINAL']}")
print(f"FILE_DATE: {sample['FILE_DATE']}       *Filled: {sample['FILE_DATE_FLAG']}*")
print(f"PERMIT_DATE: {sample['PERMIT_DATE']}   *Filled: {sample['PERMIT_DATE_FLAG']}*")
print(f"FINAL_DATE: {sample['FINAL_DATE']}     *Filled: {sample['FINAL_DATE_FLAG']}*")

print("DATES_DATA: ")
print(json.dumps(DATES_DATA, indent=2))



STATUS_NORMALIZED: Inactive    *Filled: nan*
RECORD_TYPE_ORIGINAL: Commercial - New Construction
FILE_DATE: 2013-08-09       *Filled: nan*
PERMIT_DATE: 2013-10-02   *Filled: nan*
FINAL_DATE: 2022-05-23     *Filled: nan*
DATES_DATA: 
{
  "Status": "Expired",
  "Issued Date": "Oct 2, 2013",
  "Created Date": "Aug 9, 2013",
  "Completed Date": "May 23, 2022",
  "Other Information": {
    "IssueDate": "Oct 2, 2013",
    "CreatedDate": "Aug 9, 2013",
    "CompletedDate": "May 23, 2022",
    "ExpirationDate": "Oct 2, 2014",
    "StatusDescription": "Expired"
  },
  "Construction Completed Date": null
}


In [8]:
print("DATA:")
print(json.dumps(json.loads(DATA), indent=2))



DATA:
{
  "Y": "4877167.57232901",
  "ZIP": "228",
  "\ufeffX": "-8372430.69452995",
  "STATUS": "COMPLETED",
  "ADDRESS": "20 W CHESTNUT HILL AVE",
  "OBJECTID": "171071611",
  "UNIT_NUM": "",
  "GEOCODE_X": "2678970.14882499",
  "GEOCODE_Y": "281192.96357821",
  "OPA_OWNER": "BARB DONALD E            DAVIDSON BRUCE H",
  "UNIT_TYPE": "",
  "PERMITTYPE": "ZP_ZONING",
  "TYPEOFWORK": "SFADD",
  "CENSUSTRACT": "19118-3744",
  "POSSE_JOBID": "",
  "APPLICANTTYPE": "CONTRACTOR",
  "CONTRACTORZIP": "19075-",
  "NUMBEROFUNITS": "",
  "OCCUPANCYTYPE": "",
  "PARCEL_ID_NUM": "181721",
  "USECATEGORIES": "",
  "CONTRACTORCITY": "ORELAND",
  "CONTRACTORNAME": "REGAN CONSTRUCTION CO INC",
  "MOSTRECENTINSP": "",
  "SYSTEMOFRECORD": "HANSEN",
  "ADDRESSOBJECTID": "82338",
  "CONTRACTORSTATE": "PA",
  "OPA_ACCOUNT_NUM": "092224000",
  "PERMITISSUEDATE": "2015/08/11 15:02:56+00",
  "COUNCIL_DISTRICT": "8",
  "PERMITDESCRIPTION": "ZONING PERMIT",
  "CONTRACTORADDRESS1": "320 PENNSYLVANIA AVENUE",
  